# LangSmith Evaluation with HLE Questions

## Part 1: Setup & Document Selection

This lab evaluates answers to a 20-example, text-only multiple-choice subset of Humanity's Last Exam (HLE). The dataset is gated: obtain Hugging Face access and set `HF_TOKEN` in the local `.env` file before loading examples.


In [18]:
%pip install python-dotenv langsmith openai datasets openevals


Note: you may need to restart the kernel to use updated packages.


In [19]:
from dotenv import load_dotenv
import os

load_dotenv()

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "langsmith-custom-dataset-evaluation"
os.environ["LANGSMITH_ENDPOINT"] = "https://eu.api.smith.langchain.com"

print("OpenAI key loaded:", bool(os.getenv("OPENAI_API_KEY")))
print("LangSmith key loaded:", bool(os.getenv("LANGSMITH_API_KEY")))
print("LangSmith endpoint loaded:", bool(os.getenv("LANGSMITH_ENDPOINT")))
print("Hugging Face token loaded:", bool(os.getenv("HF_TOKEN")))


OpenAI key loaded: True
LangSmith key loaded: True
LangSmith endpoint loaded: True
Hugging Face token loaded: False


In [20]:
from langsmith import Client, traceable
from langsmith.wrappers import wrap_openai
from openai import OpenAI
from openevals.llm import create_llm_as_judge
from openevals.prompts import CORRECTNESS_PROMPT
from datasets import load_dataset

client = Client()
openai_client = wrap_openai(OpenAI())

In [21]:
# Test
response = openai_client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "user",
            "content": "Reply with a short greeting and a check emoji at the end.",
        }
    ],
)

answer = response.choices[0].message.content
print(answer)

Hello! Hope you're having a great day! ✅


In [22]:
@traceable
def target_function(inputs):
    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "user", "content": inputs["question"]}
        ]
    )

    return {'answer': response.choices[0].message.content}

# Calling the decorated function creates a visible traced run.
test_result = target_function({'question': 'What is the speed of light?'})
print(test_result)

{'answer': 'The speed of light in a vacuum is approximately \\(299,792,458\\) meters per second, often rounded to \\(3.00 \\times 10^8\\) meters per second for simplicity in calculations. In terms of kilometers per second, it is approximately \\(300,000\\) km/s. This value is fundamental in physics and is denoted by the symbol \\(c\\).'}


## Part 2: Dataset Creation

#### Extract and Structure Your Examples

In [24]:
# Load text-only, multiple-choice examples
ds = load_dataset("cais/hle", split="test", streaming=True)

hle_examples = []
for sample in ds:
    if len(hle_examples) >= 20:
        break
    if sample["answer_type"] == "multipleChoice" and not sample["image"]:
        hle_examples.append({
            "question": sample["question"],
            "answer":   sample["answer"],
            "category": sample["category"],
        })

print(f"Loaded {len(hle_examples)} examples")
print("Categories:", set(e["category"] for e in hle_examples))

Loaded 20 examples
Categories: {'Humanities/Social Science', 'Computer Science/AI', 'Biology/Medicine', 'Other', 'Math'}


In [25]:
# Push to a LangSmith dataset
DATASET_NAME = "hle-mc-benchmark"

# Create dataset (skip if it already exists)
existing = [d.name for d in client.list_datasets()]
if DATASET_NAME not in existing:
    dataset = client.create_dataset(DATASET_NAME, description="HLE multiple-choice subset")
    client.create_examples(
        inputs           = [{"question": e["question"]} for e in hle_examples],
        outputs          = [{"answer":   e["answer"]}   for e in hle_examples],
        dataset_id       = dataset.id,
    )
    print(f"Created dataset with {len(hle_examples)} examples")
else:
    print(f"Dataset '{DATASET_NAME}' already exists — skipping upload")

Created dataset with 20 examples
